# Training scRepresenter

This step involves training both scNET and scGPT, and combining the resulting output. GPU usage is highly recommended.

In [1]:
#required imports and misc.

import os
import scanpy as sc
if os.path.basename(os.getcwd()) == 'scripts':
    os.chdir('..')
from src.main import run_scRepresenter

RUN_NAME = "pbmc3k"

SAVE_DIR = "output/" + RUN_NAME
os.makedirs(SAVE_DIR, exist_ok=True)

Global seed set to 0


## Prepare input data

There are two required inputs to run the scRepresenter pipeline, a scRNA-seq dataset and a gene similarity network. We already processed a scRNA-seq dataset (see the Preprocessing notebook), so we just need to read the file into memory.

For the gene network, we will use the Protein-Protein Interactions graph that we provide. For a guide on how to produce your custom network, see the Networks Section in the repository.

In [2]:
obj = sc.read("./data/pbmc3k_final.h5ad")
graph_file = 'PPI.csv'

## Model training

Optionally, the parameters of each model can be individually changed. Additionally, if no parameters are given for one of the models, it simply runs with the default values. 

In [3]:
parameters_scnet = dict(
    annotation_file=graph_file,
    pre_processing_flag=False,
    human_flag=False,
    number_of_batches=1,
    split_cells=True,
    max_epoch=300,
    model_name=RUN_NAME,
    clf_loss=False,
)

parameters_scgpt = dict(
    model_name=RUN_NAME,
    seed=0,
    dataset_name=RUN_NAME,
    do_train=True,
    test_size=0.4,
    load_model="./src/scgpt/checkpoints/scGPT_human",
    mask_ratio=0,
    epochs=10,
    n_bins=51,
    MVC=False, # Masked value prediction for cell embedding
    ecs_thres=0.0, # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,
    lr=1e-4,
    batch_size=16,
    layer_size=75,
    nlayers=4,  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
    nhead=4,  # number of heads in nn.MultiheadAttention
    dropout=0.3,  # dropout probability
    schedule_ratio=0.9,  # ratio of epochs for learning rate schedule
    save_eval_interval=5,
    fast_transformer=True,
    pre_norm=False,
    amp=True,  # Automatic Mixed Precision
    include_zero_gene = False,
    freeze = False, #freeze
    DSBN = False,  # Domain-spec batchnorm
)

In this tutorial we only use the Pbmc3k dataset, so the model will train and test on separate partitions of the data according to the *test_size* parameter. There is also the possibility of using a specific training dataset, by using the *training_obj* argument.

Finally, you can train the models by calling the following function, with the option of training both models or just one.

In [4]:
common_scnet_embs, common_scgpt_embs, avg_combined_embs, conq_combined_embs, common_labels = \
    run_scRepresenter(RUN_NAME, obj, graph_file, SAVE_DIR, 300, 10, parameters_scnet=parameters_scnet, parameters_scgpt=parameters_scgpt, training_obj=None)

         Falling back to preprocessing with `sc.pp.pca` and default params.


/app/src/scNET/PPI.csv


N genes: (690, 2638)


Highly variable genes: 690


Training:   0%|                                                         | 0/300 [00:00<?, ?it/s]

...

scGPT - INFO - -----------------------------------------------------------------------------------------


scGPT - INFO - | end of epoch  10 | time:  6.16s | valid loss/mse 0.2971 | err 0.0377


scGPT - INFO - -----------------------------------------------------------------------------------------


Retrieved gene embeddings for 1803 genes.
scGPT - INFO - Accuracy: 0.935, Precision: 0.935, Recall: 0.951, Macro F1: 0.941


Test cells in common: 1056 / 1056
